# **Muhammad Ibtisam Afzal**

# Assignment No. 11 — Microsoft Azure AI Services
## Two Python-based solutions: Image Analysis and Text Sentiment Analysis

**Selected services**
1. **Azure AI Vision — Image Analysis:** produces captions, tags, and detected objects from an image.
2. **Azure AI Language — Sentiment Analysis:** classifies text sentiment and returns confidence scores.

This notebook uses Azure REST APIs and prompts for endpoint/key credentials at runtime. Do not place subscription keys directly in notebook source code.

## 1. Create Azure AI resources using the Azure Portal

Create two resources in your Azure subscription:

### Resource A — Azure AI Vision
1. Sign in to [Azure Portal](https://portal.azure.com).
2. Search for **Computer Vision** or **Azure AI Vision**.
3. Create a resource, choose your subscription, resource group, a supported region, and an available pricing tier.
4. Deploy the resource and open **Keys and Endpoint**.
5. Copy the endpoint and one key.

### Resource B — Azure AI Language
1. In Azure Portal, search for **Language service** or **Azure AI Language**.
2. Create a Language resource with your subscription, resource group, supported region, and available pricing tier.
3. Deploy it and open **Keys and Endpoint**.
4. Copy the endpoint and one key.

The resources may have different endpoints and keys. Keep both keys private. Azure portal labels can vary slightly as Microsoft updates the interface.

## 2. Install dependencies and configure credentials

The `getpass` prompts hide API keys as you type. The endpoint is not secret, but the key is.

In [1]:
import sys
!{sys.executable} -m pip -q install requests

import requests
import json
from getpass import getpass
from urllib.parse import urlparse

print("Requests library is ready.")

Requests library is ready.


In [2]:
# Azure AI Vision credentials
vision_endpoint = input("Paste Azure AI Vision endpoint: ").strip().rstrip("/")
vision_key = getpass("Paste Azure AI Vision key (hidden): ").strip()

# Azure AI Language credentials
language_endpoint = input("Paste Azure AI Language endpoint: ").strip().rstrip("/")
language_key = getpass("Paste Azure AI Language key (hidden): ").strip()

for label, value in [
    ("Vision endpoint", vision_endpoint),
    ("Language endpoint", language_endpoint)
]:
    if not value.startswith("https://"):
        raise ValueError(f"{label} must start with https://")

if not vision_key or not language_key:
    raise ValueError("Both Azure resource keys are required.")

print("Both endpoints accepted; both keys received securely.")

Paste Azure AI Vision endpoint: https://navttc-vision.cognitiveservices.azure.com/
Paste Azure AI Vision key (hidden): ··········
Paste Azure AI Language endpoint: https://lang-navttc-mlops-69de5.cognitiveservices.azure.com/
Paste Azure AI Language key (hidden): ··········
Both endpoints accepted; both keys received securely.


## 3. Service 1 — Azure AI Vision: Image Analysis

### Problem solved
Image Analysis extracts information from visual content. It can generate a natural-language caption, identify tags, and detect objects. This helps applications understand images without requiring a person to manually describe every picture.

### Python integration
The request below uses the Image Analysis 4.0 REST API. The image is supplied as a publicly accessible URL. The request is authenticated with the Vision resource key in the `Ocp-Apim-Subscription-Key` header.

In [3]:
VISION_API_VERSION = "2024-02-01"

def analyze_image(image_url):
    if not image_url.startswith(("https://", "http://")):
        raise ValueError("Provide a publicly accessible image URL.")

    url = (
        f"{vision_endpoint}/computervision/imageanalysis:analyze"
        f"?api-version={VISION_API_VERSION}"
        "&features=caption,tags,objects"
        "&language=en"
    )
    headers = {
        "Ocp-Apim-Subscription-Key": vision_key,
        "Content-Type": "application/json"
    }
    payload = {"url": image_url}

    response = requests.post(url, headers=headers, json=payload, timeout=90)
    if not response.ok:
        raise RuntimeError(
            f"Azure Vision error ({response.status_code}): {response.text}"
        )
    return response.json()

# Microsoft Learn sample image; replace with another publicly accessible image if desired.
sample_image_url = (
    "https://learn.microsoft.com/azure/ai-services/computer-vision/"
    "media/quickstarts/presentation.png"
)

print("INPUT IMAGE URL:", sample_image_url)
vision_result = analyze_image(sample_image_url)
print("\\nAZURE AI VISION RESPONSE:")
print(json.dumps(vision_result, indent=2))

INPUT IMAGE URL: https://learn.microsoft.com/azure/ai-services/computer-vision/media/quickstarts/presentation.png
\nAZURE AI VISION RESPONSE:
{
  "modelVersion": "2023-10-01",
  "captionResult": {
    "text": "a man pointing at a screen",
    "confidence": 0.7766579389572144
  },
  "metadata": {
    "width": 1038,
    "height": 692
  },
  "tagsResult": {
    "values": [
      {
        "name": "text",
        "confidence": 0.9966012835502625
      },
      {
        "name": "clothing",
        "confidence": 0.9801061749458313
      },
      {
        "name": "person",
        "confidence": 0.9596298933029175
      },
      {
        "name": "display device",
        "confidence": 0.9490274786949158
      },
      {
        "name": "indoor",
        "confidence": 0.947483241558075
      },
      {
        "name": "wall",
        "confidence": 0.9395942091941833
      },
      {
        "name": "media",
        "confidence": 0.9306115508079529
      },
      {
        "name": "television

### Interpreting the Image Analysis output

Look for these response fields:
- `captionResult`: a short description of the image and a confidence score.
- `tagsResult.values`: detected visual tags and confidence scores.
- `objectsResult.values`: detected objects, their bounding boxes, and associated tags.

The service response depends on the image content and model output. Keep the actual response produced by your Azure resource as the assignment evidence.

## 4. Service 2 — Azure AI Language: Sentiment Analysis

### Problem solved
Sentiment Analysis evaluates text and estimates whether the expressed opinion is positive, negative, neutral, or mixed. It also returns confidence scores for the sentiment classes.

### Python integration
The REST API request sends one or more text documents to the Language resource. Authentication uses the Language resource key in the `Ocp-Apim-Subscription-Key` header.

In [4]:
LANGUAGE_API_VERSION = "2024-11-01"

def analyze_sentiment(text, language="en"):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Provide non-empty text.")

    url = (
        f"{language_endpoint}/language/:analyze-text"
        f"?api-version={LANGUAGE_API_VERSION}"
    )
    headers = {
        "Ocp-Apim-Subscription-Key": language_key,
        "Content-Type": "application/json"
    }
    payload = {
        "kind": "SentimentAnalysis",
        "analysisInput": {
            "documents": [
                {"id": "1", "language": language, "text": text}
            ]
        },
        "parameters": {
            "modelVersion": "latest"
        }
    }

    response = requests.post(url, headers=headers, json=payload, timeout=90)
    if not response.ok:
        raise RuntimeError(
            f"Azure Language error ({response.status_code}): {response.text}"
        )
    return response.json()

sample_text = (
    "The new online learning platform is easy to use. "
    "The lessons are helpful and the support team responds quickly."
)

print("INPUT TEXT:")
print(sample_text)
language_result = analyze_sentiment(sample_text)
print("\\nAZURE AI LANGUAGE RESPONSE:")
print(json.dumps(language_result, indent=2))

INPUT TEXT:
The new online learning platform is easy to use. The lessons are helpful and the support team responds quickly.
\nAZURE AI LANGUAGE RESPONSE:
{
  "kind": "SentimentAnalysisResults",
  "results": {
    "documents": [
      {
        "id": "1",
        "sentiment": "positive",
        "confidenceScores": {
          "positive": 0.91,
          "neutral": 0.09,
          "negative": 0.0
        },
        "sentences": [
          {
            "sentiment": "positive",
            "confidenceScores": {
              "positive": 0.82,
              "neutral": 0.18,
              "negative": 0.0
            },
            "offset": 0,
            "length": 49,
            "text": "The new online learning platform is easy to use. "
          },
          {
            "sentiment": "positive",
            "confidenceScores": {
              "positive": 1.0,
              "neutral": 0.0,
              "negative": 0.0
            },
            "offset": 49,
            "length": 62,

### Interpreting the Sentiment Analysis output

Inspect `results.documents[0]` in the response:
- `sentiment`: predicted overall sentiment (positive, negative, neutral, or mixed).
- `confidenceScores`: confidence estimates for positive, neutral, and negative sentiment.
- `sentences`: sentence-level sentiment and confidence scores, when returned.

Confidence scores are model estimates, not guarantees. Test the service with different examples to see how the output changes.

## 5. Real-world applications of both services

### Azure AI Vision — Retail inventory and accessibility
A retail application can analyze product photographs to generate captions and tags, helping organize a visual catalog and improve image search. Caption generation can also support accessibility by providing descriptions for users who rely on screen readers.

### Azure AI Language — Customer feedback analysis
A company can analyze customer reviews, survey responses, and support messages to estimate customer sentiment. Teams can monitor changes in satisfaction, prioritize negative feedback, and identify service issues for follow-up.

## 6. Limitations and challenges

- **Image Analysis:** image quality, lighting, occlusion, unusual viewpoints, or visually similar objects can reduce accuracy. The service may miss objects or generate incomplete captions.
- **Sentiment Analysis:** sarcasm, idioms, mixed opinions, domain-specific language, and cultural context can lead to incorrect classifications. A sentiment score does not explain every nuance of a customer's meaning.

For both services, validate results on representative data and retain human review for important decisions.

## 7. Submission evidence — screenshots to capture

Capture these screenshots from your own Azure account and executed notebook:

1. **Vision resource overview** — Azure Portal showing the deployed Vision resource.
2. **Language resource overview** — Azure Portal showing the deployed Language resource.
3. **Vision Keys and Endpoint** — show the endpoint if required, but hide/blur the key.
4. **Language Keys and Endpoint** — show the endpoint if required, but hide/blur the key.
5. **Image Analysis result** — notebook cell showing the sample image URL and actual JSON response, including caption/tags/objects where returned.
6. **Sentiment Analysis result** — notebook cell showing the sample text and actual sentiment/confidence response.

Never submit screenshots that expose API keys. The notebook hides key input using `getpass`.

## 8. Conclusion

This assignment demonstrates two Azure AI services through working Python REST API patterns:

1. Azure AI Vision Image Analysis accepts an image URL and returns visual information such as captions, tags, and objects.
2. Azure AI Language Sentiment Analysis accepts text and returns sentiment labels with confidence scores.

Both examples use resource-specific endpoints and subscription-key authentication. The actual API responses and screenshots must be generated by running the notebook with your own Azure resources.